About this notebook.

This notebook goes through all the texts and aplies on them a NLP pipeline consisting of (1) cleaning of the raw text, (2) sentence tokenization, (3) part-of-speech annotation, (4) lemmatization, and (5) named entity recognition.

The processed textual data are saved for future reuse.

In [1]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd
import json

/home/jupyter-vojta/notebooks/labyrinth/venv_torch_nlp/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
emlap_metadata = pd.read_csv("../data/emlap_metadata.csv", sep=";", index_col=0)
emlap_metadata.head(5)

,working_title,filenames,no.,is_done,is_noscemus,if_noscemus_id,AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,CONTENTS,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,other_notes,tokens_N,aurhor_wd
0,"Augurello, Chrysopoeia",100001_Augurello1515_Chrysopoeia_GB_Noscemus,100001,True,True,713324.0,NaN,True,NaN,True,...,NaN,didactic poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,GB,Noscemus,NaN,23718,NaN
1,"Pseudo-Lull, Secretis",100002_Pseudo-Lull1518_De secretis_naturae_MDZ...,100002,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,"alchemy, medicine",NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,24673,NaN
2,"Pantheus, Ars Transmutatione",100003_Pantheus1518_Ars_Transmutationis_Metall...,100003,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,alchemy,NaN,https://www.google.co.uk/books/edition/Ars_Tra...,GB,BL,NaN,8646,NaN
3,"Anon, Vera alchemiae",100004_Anon1561_Verae_Alchemiae_MDZ_MBS,100004,True,False,NaN,NaN,True,NaN,True,...,NaN,"compendium, florilegium",alchemy,NaN,https://mdz-nbn-resolving.de/details:bsb10141168,MDZ,MBS,NaN,3521,NaN
4,"Pantheus, Voarchadumia",100005_Pantheus1530_Voarchadumia_ONB,100005,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,alchemy,NaN,https://data.onb.ac.at/rep/10588E49,ONB,ONB,NaN,20386,NaN


In [5]:
row = emlap_metadata.loc[emlap_metadata["no."]==100001].iloc[0]

In [6]:
row["working_title"]

'Augurello, Chrysopoeia'

In [7]:
import json
import ast
import re

def parse_messy_json(text):
    if not isinstance(text, str):
        return None

    # 1. Try strict JSON first
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2. Fix common issues
    fixed = text.strip()

    # Replace single quotes with double quotes when appropriate
    fixed = fixed.replace("'", '"')

    # Remove trailing commas before ] or }
    fixed = re.sub(r",\s*([}\]])", r"\1", fixed)

    # Ensure keys are quoted (naively, but works for your case)
    fixed = re.sub(r"(?<=\{|\s)([A-Za-z_][A-Za-z0-9_]*)(?=\s*:)", r'"\1"', fixed)

    # 3. Try JSON again
    try:
        return json.loads(fixed)
    except Exception:
        pass

    # 4. Try Python literal (safer than eval)
    try:
        return ast.literal_eval(text)
    except Exception:
        pass

    # 5. Give up
    return None

In [8]:
emlap_metadata["if_compendium_parsed"] = emlap_metadata["if_compendium"].apply(parse_messy_json).tolist()

In [9]:
len(emlap_metadata)

100

In [10]:
#filename_id_dict = dict(zip(emlap_metadata["filename"], emlap_metadata["No."]))

For preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level one level up.

The module can be clonned from here: https://github.com/CCS-ZCU/latin-preprocessing and imported to python following the steps below:

In [40]:
# for preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level as the current project.
current_working_directory = os.getcwd()
relative_path = '../../latin-preprocessing/' # change according to your location...
module_path = os.path.abspath(os.path.join(current_working_directory, relative_path))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
# Now import the module
import tomela

In [128]:
importlib.reload(tomela)

Using CPU for spaCy.


<module 'tomela' from '/home/jupyter-vojta/notebooks/latin-preprocessing/tomela/__init__.py'>

In [129]:
tomela.nlp.pipeline

[('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x7a92a40ceb10>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7a9200360770>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7a9200360590>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x7a92003609b0>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x7a9200360470>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7a92003076f0>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x7a90e62e84a0>),
 ('remorpher', <function la_core_web_lg.functions.remorpher(doc)>)]

In [130]:
tomela.nlp.max_length = 4000000

In [14]:
doc = tomela.nlp("Veritas, vt vlla dicit, semper est universalis et a principiis fundamentalis oritur (lib. 3, cap. VI)")
for token in doc:
    print((token.text, token.lemma_, token.pos_))

('Veritas', 'ueritas', 'NOUN')
(',', ',', 'PUNCT')
('vt', 'vt', 'ADV')
('vlla', 'vllus', 'NOUN')
('dicit', 'dico', 'VERB')
(',', ',', 'PUNCT')
('semper', 'semper', 'ADV')
('est', 'sum', 'AUX')
('universalis', 'uniuersalis', 'ADJ')
('et', 'et', 'CCONJ')
('a', 'ab', 'ADP')
('principiis', 'principium', 'NOUN')
('fundamentalis', 'fundamentalis', 'ADJ')
('oritur', 'orior', 'VERB')
('(lib', '(lib', 'NOUN')
('.', '.', 'PUNCT')
('3', '3', 'NUM')
(',', ',', 'PUNCT')
('cap', 'capitulum', 'NOUN')
('.', '.', 'PUNCT')
('VI', 'uis', 'NUM')
(')', ')', 'PUNCT')


In [15]:
source_path = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/"
len(os.listdir(source_path))

200

In [16]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/", "../data/emlap_annotated_textblocks/", dirs_exist_ok=True)

'../data/emlap_annotated_textblocks/'

In [17]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['100001_Augurello1515_Chrysopoeia_GB_Noscemus.json',
 '100002_Pseudo-Lull1518_De_secretis_naturae_MDZ_MBS.json',
 '100003_Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 '100004_Anon1561_Verae_Alchemiae_MDZ_MBS.json',
 '100005_Pantheus1530_Voarchadumia_ONB.json',
 '100006_Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json',
 '100007_Anon1550_Rosarium_philosophorum_ER_ZZ.json',
 '100008_Severinus1572_Epistola_MBZ_MBS.json',
 '100009_Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json',
 '100010_Bracesco1548_De_alchemia_dialogi_duo_IA_Madrid.json',
 '100011_Anon1541_De_alchemia_MDZ_MBS.json',
 '100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100014_Toxites1567_Spongia_stibii_MDZ_MBS.json',
 '100015_Gessner1569_Thesaurus_Euonymi_Philiatri_liber_secundus_MDZ_MBS.json',
 '100016_Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 '100017_Bodenstein1559_Isagoge_MDZ_MBS.json',
 '100018_Trevisanus1567_Pe

In [18]:
row = emlap_metadata.loc[emlap_metadata["no."]==int("100001")].iloc[0]
row

working_title                                                                                 Augurello, Chrysopoeia
filenames                                                               100001_Augurello1515_Chrysopoeia_GB_Noscemus
no.                                                                                                           100001
is_done                                                                                                         True
is_noscemus                                                                                                     True
if_noscemus_id                                                                                              713324.0
AUTHORSHIP                                                                                                       NaN
is_one_author                                                                                                   True
#if more than 1 author skip section and choose compendium below 

In [19]:
files_overview = []
for filename in os.listdir(source_path):
    #filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
    id = int(filename[:6])
    row = emlap_metadata.loc[emlap_metadata["no."]==id].iloc[0]
    if "_params" not in filename:
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        pages_n = len(textblocks)
        chars_n = sum([sum([len(tb["text"]) for tb in p]) for p in textblocks])
        files_overview.append({"filename" : filename, "pages_n" : pages_n, "chars_n" : chars_n})
files_processed = pd.DataFrame(files_overview)
files_processed

,filename,pages_n,chars_n
0,100084_Croll1609_Basilica_chymica_MDZ_MBS.json,477,694793
1,100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json,508,694493
2,100058_Hagecius1596_Actio_medica_ER_UBB.json,89,104677
3,100013_Ulstad1525_Coelum_philosophorum_Medica_...,113,237793
4,100069_Severinus1571_Idea_medicinae_philosophi...,463,600046
...,...,...,...
95,100056_Claveus1598_Apologia_crysopoeiae_MDZ_MB...,233,233328
96,100065_Libavius1594_Neoparacelsica_MDZ_MBS.json,821,1079977
97,100031_Phaedro1562_Aquila_coelestis_MBZ_MBS.json,55,15562
98,100098_Burggravius1630_Biolychnium_VD17_SLUB.json,167,186267


In [20]:
#emlap_catalogue = google_conf.setup(sheet_url="https://docs.google.com/spreadsheets/d/1bkHHTYc86K2IuEXqfYfkDNt5LovtvCU3gvqHIbVio88/edit?usp=sharing", service_account_path="../../../ServiceAccountsKey.json")

# google_conf.set_with_dataframe(emlap_catalogue.add_worksheet("files_processed", 1,1), files_processed, include_index=False)


Develop and test with one example test

In [21]:
filename = '100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [22]:
len(textblocks)

413

In [ ]:
textblocks[30][:10]

In [25]:
for p in textblocks[:15]:
    for t in p:
        if "[" in t["text"]:
            print(t)

{'coordinates': [384.0, 2262.9599609375, 2165.489013671875, 2267.760009765625], 'text': '[GR]χιμικώτερα[/GR] essent seligere, dissentientiaquè componere. Itaque & nouam\n', 'tag': 'text'}
{'coordinates': [139.1999969482422, 3188.159912109375, 1914.183349609375, 3192.9599609375], 'text': 'quoddam [GR]μιγμα[/GR] & commentum ab omni veritate alienum. Sed ostendendum est veram artem nec\n', 'tag': 'text'}
{'coordinates': [398.1600036621094, 511.6800231933594, 2167.622314453125, 516.4800415039062], 'text': 'longe abest à [GR]πανσπερμία[/GR] illa vetere, & atomis Democriti, & multiplici vicissim oriente & pereunte? Sed\n', 'tag': 'text'}
{'coordinates': [390.7200012207031, 2782.079833984375, 2149.680419921875, 2786.8798828125], 'text': 'quętis [GR]σκευασταί καὶ πυροτεηνίαν[/GR], ecce tibi eundem Bulcasim passim inseruientem. Fornaces enim seu Atha¬\n', 'tag': 'text'}


In [136]:
for p in textblocks:
    for t in p:
        if "[/S]" in t["text"]:
            print(t)

{'coordinates': [225.83999633789062, 905.7598266601562, 387.3038635253906, 910.5598754882812], 'text': 'Olei [S]y[/S]. j. ex\n', 'tag': 'margin'}
{'coordinates': [429.1199951171875, 1108.5599365234375, 2191.517578125, 1113.35986328125], 'text': 'crassos. Dosis eius solius [S]z[/S] ij. vsque ad [S]z[/S] iiij. cum aqua frigida. Similia habet Mesues lib simplicium, cap. 16.\n', 'tag': 'text'}
{'coordinates': [241.9199981689453, 2369.999755859375, 407.46197509765625, 2374.7998046875], 'text': 'trahe [S][/S], ex\n', 'tag': 'margin'}
{'coordinates': [234.72000122070312, 2417.999755859375, 407.2383728027344, 2422.7998046875], 'text': 'hoc\xa0[S][/S], & i¬\n', 'tag': 'margin'}
{'coordinates': [424.0799865722656, 312.7200622558594, 2143.775634765625, 317.5200500488281], 'text': 'Lunam, Venerem, Martem, Saturnum, louem; signare vero more astrologorum,\xa0[S]Mercurius[/S],\xa0[S]Sol[/S],\xa0[S]Luna[/S],\xa0[S]Venus[/S],\xa0[S]Mars[/S],\xa0[S]Saturnus[/S],\xa0[S]Jupiter[/S].\n', 'tag': 'text'}
{'c

In [26]:
def add_tag_info_to_char_mapping(full_text, char_to_source):
    """
    Parse inline tags like [GR]...[/GR], [M]...[/M], [S]...[/S] in full_text
    and attach a "tags" field (list of active tags) to char_to_source[char_idx].
    """
    active_tags = []  # stack / list of currently open tags
    i = 0
    n = len(full_text)

    while i < n:
        ch = full_text[i]

        if ch == "[":
            # Try to find the closing bracket of this [TAG] or [/TAG]
            close = full_text.find("]", i + 1)
            if close == -1:
                # malformed tag, treat as normal char
                if i in char_to_source and active_tags:
                    tags_set = char_to_source[i].setdefault("tags", set())
                    tags_set.update(active_tags)
                i += 1
                continue

            content = full_text[i + 1:close]

            if content.startswith("/"):
                # closixng tag, e.g. [/GR]
                label = content[1:]
                # remove from active_tags (LIFO or first match)
                if active_tags and active_tags[-1] == label:
                    active_tags.pop()
                elif label in active_tags:
                    active_tags.remove(label)
                # We don't assign tags to the '['...']' characters themselves here
            else:
                # opening tag, e.g. [GR]
                label = content
                active_tags.append(label)

            # skip past the closing ']'
            i = close + 1
            continue

        else:
            # normal character: assign currently active tags (if any)
            if i in char_to_source and active_tags:
                tags_set = char_to_source[i].setdefault("tags", set())
                tags_set.update(active_tags)
            i += 1

    # normalize sets → lists (so they are JSON-serializable, nicer to inspect)
    for info in char_to_source.values():
        if "tags" in info and isinstance(info["tags"], set):
            info["tags"] = sorted(info["tags"])

    return char_to_source

In [137]:
import re
import unicodedata
from spacy.tokens import Token, Doc
from spacy.language import Language


# -------------------------------------------------------------------
# 0. Token / Doc extensions
# -------------------------------------------------------------------
for ext, default in [
    ("pages", None),
    ("textblocks", None),
    ("tags", None),
    ("block_type", None),
]:
    if not Token.has_extension(ext):
        Token.set_extension(ext, default=default)

if not Doc.has_extension("char_to_source"):
    Doc.set_extension("char_to_source", default=None)


# -------------------------------------------------------------------
# CLEANER: keep tags exact, clean only outside-tag regions
# -------------------------------------------------------------------
def text_cleaner_keep_tags(raw_text):
    """
    Clean Latin text but NEVER modify anything inside markup spans like:
        [GR]...[/GR], [S]...[/S], [M]...[/M]

    Behavior:
      - Split raw text into TAG / NON-TAG parts
      - Maintain a stack (to allow nested tags)
      - CLEAN only outside-tag parts
      - Inside-tag content is preserved exactly and NFC-normalized
    """

    parts = re.split(r"(\[[A-Za-z]+]|\[/[A-Za-z]+])", raw_text)
    cleaned_parts = []
    stack = []

    for part in parts:

        # Opening tag
        if re.fullmatch(r"\[[A-Za-z]+]", part):
            stack.append(part[1:-1])
            cleaned_parts.append(part)
            continue

        # Closing tag
        if re.fullmatch(r"\[/[A-Za-z]+]", part):
            name = part[2:-1]
            if stack and stack[-1] == name:
                stack.pop()
            cleaned_parts.append(part)
            continue

        # Inside a tag span → do not modify, only normalize
        if stack:
            cleaned_parts.append(unicodedata.normalize("NFC", part))
            continue

        # Outside tags → apply Latin cleaning
        x = part
        x = x.replace("¬\n", "").replace("\n", " ")
        x = x.replace("ß", "ss").replace("ij", "ii")
        x = re.sub(r"\s\s+", " ", x)
        x = re.sub(r'\b(\w)(\w*)\b', lambda m: m.group(1) + m.group(2).lower(), x)
        x = x.replace(". &", ", &")
        x = x.replace("v", "u").replace("V", "U")

        cleaned_parts.append(x)

    return "".join(cleaned_parts)


# -------------------------------------------------------------------
# 1. Build raw + clean-with-tags + mapping
# -------------------------------------------------------------------
def process_textblocks(textblocks):
    raw_full = ""
    clean_full = ""
    raw_to_clean = {}
    char_src = {}

    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):

            if tb["tag"] not in {"text", "title", "margin"}:
                continue

            raw_text = tb["text"]
            clean_text = text_cleaner_keep_tags(raw_text)

            r0 = len(raw_full)
            c0 = len(clean_full)

            r_i = c_i = 0
            while r_i < len(raw_text) and c_i < len(clean_text):

                rc = raw_text[r_i]
                cc = clean_text[c_i]

                raw_idx = r0 + r_i
                clean_idx = c0 + c_i

                if rc == cc:
                    raw_to_clean[raw_idx] = clean_idx
                    char_src[clean_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx,
                        "textblock_type": tb["tag"],
                    }
                    r_i += 1
                    c_i += 1
                    continue

                # raw has markup [TAG]
                if raw_text.startswith("[", r_i):
                    end = raw_text.find("]", r_i)
                    if end == -1:
                        # malformed → treat as normal
                        raw_to_clean[raw_idx] = clean_idx
                        char_src[clean_idx] = {
                            "page_idx": page_idx,
                            "textblock_idx": tb_idx,
                            "textblock_type": tb["tag"],
                        }
                        r_i += 1
                        c_i += 1
                    else:
                        # skip tag entirely in raw
                        for skip in range(r_i, end + 1):
                            raw_to_clean[r0 + skip] = None
                        r_i = end + 1
                    continue

                # cleaned version differs (ASCII conversion etc.)
                raw_to_clean[raw_idx] = clean_idx
                char_src[clean_idx] = {
                    "page_idx": page_idx,
                    "textblock_idx": tb_idx,
                    "textblock_type": tb["tag"],
                }
                r_i += 1
                c_i += 1

            # remaining raw chars (rare)
            while r_i < len(raw_text):
                raw_to_clean[r0 + r_i] = None
                r_i += 1

            raw_full += raw_text
            clean_full += clean_text

    return raw_full, clean_full, raw_to_clean, char_src


# -------------------------------------------------------------------
# 2. Greek-aware tag extractor
# -------------------------------------------------------------------
GREEK_RE = re.compile(r"[\u0370-\u03FF\u1F00-\u1FFF]")

def extract_tags_from_raw(raw_text):
    """
    Extract tags with Greek awareness:

    RULES:
      - Explicit [GR] ... [/GR] → exact span
      - Lone [GR] without closing → auto-close when Greek stops
      - Greek without tags → implicit GR tag on those chars
    """

    raw_tags = {}
    markup_spans = []

    TAG = re.compile(r"\[(?P<tag>[A-Za-z]+)]|\[/(?P<etag>[A-Za-z]+)]")
    matches = list(TAG.finditer(raw_text))

    # record tag token spans
    for m in matches:
        markup_spans.append(m.span())

    # explicit [GR] ... [/GR]
    open_pos = None
    for m in matches:
        start, end = m.span()
        tag = m.group("tag")
        etag = m.group("etag")

        if tag == "GR":
            open_pos = end
        elif etag == "GR" and open_pos is not None:
            for i in range(open_pos, start):
                if GREEK_RE.search(raw_text[i]):
                    raw_tags.setdefault(i, set()).add("GR")
            open_pos = None

    # unclosed [GR] → close at end of Greek run
    if open_pos is not None:
        i = open_pos
        while i < len(raw_text) and GREEK_RE.search(raw_text[i]):
            raw_tags.setdefault(i, set()).add("GR")
            i += 1

    # implicit GR (Greek outside tags)
    for idx, ch in enumerate(raw_text):
        if GREEK_RE.search(ch):
            raw_tags.setdefault(idx, set()).add("GR")

    return raw_tags, markup_spans


# -------------------------------------------------------------------
# 3. raw → clean tag mapping
# -------------------------------------------------------------------
def map_raw_tags_to_clean(raw_tags, raw_to_clean):
    clean_tags = {}
    for raw_idx, tagset in raw_tags.items():
        clean_idx = raw_to_clean.get(raw_idx)
        if clean_idx is not None:
            clean_tags.setdefault(clean_idx, set()).update(tagset)
    return clean_tags


# -------------------------------------------------------------------
# 4. remove markup from cleaned Latin text
# -------------------------------------------------------------------
def remove_markup(clean_text):
    return re.sub(r"\[[A-Za-z]+\]|\[/[A-Za-z]+\]", "", clean_text)


# -------------------------------------------------------------------
# 5. High-level processor
# -------------------------------------------------------------------
def process_with_source_tracking(textblocks, nlp):

    raw_text, clean_with_tags, raw_to_clean, char_src = process_textblocks(textblocks)

    raw_tag_map, _ = extract_tags_from_raw(raw_text)

    clean_tag_map = map_raw_tags_to_clean(raw_tag_map, raw_to_clean)

    clean_text_final = remove_markup(clean_with_tags)

    # inject tag info into char_src
    for clean_idx, tagset in clean_tag_map.items():
        if clean_idx in char_src:
            char_src[clean_idx]["tags"] = sorted(tagset)

    doc = nlp.make_doc(clean_text_final)
    doc._.char_to_source = char_src

    for name, proc in nlp.pipeline:
        doc = proc(doc)

    return doc


# -------------------------------------------------------------------
# 6. Source tracker: assign metadata to tokens
# -------------------------------------------------------------------
@Language.component("source_tracker")
def source_tracker(doc):
    cts = doc._.char_to_source
    if cts is None:
        return doc

    for token in doc:
        pages, tb, blocktypes, tags = set(), set(), set(), set()

        for i in range(token.idx, token.idx + len(token.text)):
            info = cts.get(i)
            if not info:
                continue
            pages.add(info["page_idx"])
            tb.add(info["textblock_idx"])
            blocktypes.add(info["textblock_type"])
            if "tags" in info:
                tags.update(info["tags"])

        token._.pages = sorted(pages) if pages else None
        token._.textblocks = sorted(tb) if tb else None
        token._.block_type = next(iter(blocktypes)) if blocktypes else "text"
        token._.tags = sorted(tags) if tags else None

    return doc


# -------------------------------------------------------------------
# 7. blocktype-based sentence splitter
# -------------------------------------------------------------------
@Language.component("blocktype_sentencizer")
def blocktype_sentencizer(doc):
    prev_bt = None
    for token in doc:
        bt = token._.block_type
        if prev_bt is None or bt != prev_bt:
            token.is_sent_start = True
        prev_bt = bt
    return doc


# -------------------------------------------------------------------
# Insert components into pipeline
# -------------------------------------------------------------------
nlp = tomela.nlp  # your existing model

if "source_tracker" in nlp.pipe_names:
    nlp.remove_pipe("source_tracker")
nlp.add_pipe("source_tracker", before="senter")

if "blocktype_sentencizer" in nlp.pipe_names:
    nlp.remove_pipe("blocktype_sentencizer")
nlp.add_pipe("blocktype_sentencizer", after="source_tracker")

<function __main__.blocktype_sentencizer(doc)>

In [ ]:
doc = process_with_source_tracking(textblocks, tomela.nlp)

In [133]:
for t in doc:
    if t._.tags:
        print("TAG:", repr(t.text), t._.tags)

TAG: 'χιμικωτερα' ['GR']
TAG: 'essent' ['GR']
TAG: 'commentum' ['GR']
TAG: ',' ['GR']
TAG: '&' ['GR']
TAG: 'atomis' ['GR']
TAG: 'tibi' ['GR']
TAG: 'eundem' ['GR']
TAG: 'Bulcasim' ['GR']
TAG: 'passim' ['GR']
TAG: 'mores' ['GR']
TAG: 'thessalicos' ['GR']
TAG: 'libera' ['GR']
TAG: ',' ['GR']
TAG: 'si' ['GR']
TAG: 'potes' ['GR']
TAG: 'prosyllogismus' ['GR']
TAG: 'aeque' ['GR']
TAG: 'Discat' ['GR']
TAG: 'Censor' ['GR']
TAG: 'specialium' ['GR']
TAG: 'extractum' ['GR']
TAG: '.' ['GR']
TAG: 'In' ['GR']
TAG: 'hoc' ['GR']
TAG: 'est' ['GR']
TAG: 'ασυμβλητον' ['GR']
TAG: 'maxime' ['GR']
TAG: 'nutritur' ['GR']
TAG: 'Et' ['GR']
TAG: 'in' ['GR']
TAG: 'quisquis' ['GR']
TAG: 'hoc' ['GR']
TAG: 'statuit' ['GR']
TAG: ',' ['GR']
TAG: 'nec' ['GR']
TAG: 'ab' ['GR']
TAG: 'habitus' ['GR']
TAG: '.' ['GR']
TAG: 'Si' ['GR']
TAG: 'cognitio' ['GR']
TAG: 'καθ' ['GR']
TAG: "'" ['GR']
TAG: 'εκαστα' ['GR']
TAG: 'eodem' ['GR']
TAG: 'explorationes' ['GR']
TAG: 'inuenerunt' ['GR']
TAG: 'omne' ['GR']
TAG: 'compositum' ['GR

In [120]:
doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)), {"page" : t._.pages, "texblock" : t._.textblocks, "tags": t._.tags, "blocktype" : t._.block_type}) for t in sent]) for sent in doc.sents]
sent_data_updated = []
for n_sent, sent_data in enumerate(doc_sentdata):
    sent_data_updated.append((filename[:6], n_sent, sent_data[0], sent_data[1]))

In [121]:
sent_data_updated[100:110]

[('100085',
  100,
  'Non alium liquorem uesica reddunt.',
  [('Non',
    'non',
    'PART',
    (0, 3),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('alium',
    'alius',
    'DET',
    (4, 9),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('liquorem',
    'liquor',
    'NOUN',
    (10, 18),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('uesica',
    'uesica',
    'NOUN',
    (19, 25),
    {'page': [7], 'texblock': [23], 'tags': None, 'blocktype': 'text'}),
   ('reddunt',
    'reddo',
    'VERB',
    (26, 33),
    {'page': [7], 'texblock': [23, 24], 'tags': None, 'blocktype': 'text'}),
   ('.',
    '.',
    'PUNCT',
    (33, 34),
    {'page': [7], 'texblock': [24], 'tags': None, 'blocktype': 'text'})]),
 ('100085',
  101,
  'Non habent alia membra, quam nos, non sudores alios.',
  [('Non',
    'non',
    'PART',
    (0, 3),
    {'page': [7], 'texblock': [24], 'tags': None, 'blocktype': 'text'}

In [122]:
sents_tags = []

for sent in sent_data_updated:
    for token in sent[3]:
        blocktype = token[4]["blocktype"]
        tag = token[4]["tags"]
        if "GR" in tag:
        #if blocktype == "margin":
            sents_tags.append(sent)
            break   # go to next sentence

TypeError: argument of type 'NoneType' is not iterable

In [113]:
len(sents_tags)

19732

In [114]:
sents_tags[:5]

[('100085',
  84,
  'Studuerum aliqua probabili specie donare, & recisis neglectisque plurimis uanis, quae χιμικωτερα essent seligere, dissentientiaque componere.',
  [('Studuerum',
    'studuerum',
    'ADJ',
    (0, 9),
    {'page': [6], 'texblock': [25], 'tags': None, 'blocktype': 'text'}),
   ('aliqua',
    'aliquis',
    'PRON',
    (10, 16),
    {'page': [6], 'texblock': [25, 26], 'tags': None, 'blocktype': 'text'}),
   ('probabili',
    'probabilis',
    'ADJ',
    (17, 26),
    {'page': [6], 'texblock': [26], 'tags': None, 'blocktype': 'text'}),
   ('specie',
    'species',
    'NOUN',
    (27, 33),
    {'page': [6], 'texblock': [26], 'tags': None, 'blocktype': 'text'}),
   ('donare',
    'dono',
    'VERB',
    (34, 40),
    {'page': [6], 'texblock': [26], 'tags': None, 'blocktype': 'text'}),
   (',',
    ',',
    'PUNCT',
    (40, 41),
    {'page': [6], 'texblock': [26], 'tags': None, 'blocktype': 'text'}),
   ('&',
    '&',
    'PUNCT',
    (42, 43),
    {'page': [6], 'texbl

In [ ]:
    target_path = "/srv/data/tome/tome-corpus/sents_data_id_jsons_v4-0/"
os.makedirs(target_path, exist_ok=True)

In [ ]:
# filename_id_dict.items()

In [ ]:
source_path = "../data/emlap_annotated_textblocks/"
len(os.listdir(source_path)) # 100 for textblocks, 100 for parameters used for their extraction

In [ ]:
os.listdir(source_path)

In [ ]:
for filename in os.listdir(source_path):
    if "_params" not in filename:
            id = filename[:6]
            try:
                if id + ".json" not in os.listdir(target_path):
                    # filename = filename.replace(".pdf", ".json")
                    filepath = os.path.join(source_path, filename)
                    with open(filepath, 'r', encoding='utf-8') as f:
                            textblocks_pages = json.load(f)
                    print("currently processing: ", filename)
                    doc = process_with_source_tracking(textblocks_pages, tomela.nlp)
                    doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)),  {"page" : t._.pages, "texblock" : t._.textblocks}) for t in sent]) for sent in doc.sents]
                    sent_data_updated = []
                    for n_sent, sent_data in enumerate(doc_sentdata):
                            sent_data_updated.append((id, n_sent, sent_data[0], sent_data[1]))
                    with open(target_path + str(id) + ".json", "w") as f:
                            json.dump(sent_data_updated, f)
            except:
                print("failed with file: ", id, filename)
                pass

In [ ]:
fns_jsons = os.listdir(target_path)
fns_jsons[:10]

In [ ]:
len(fns_jsons)

In [ ]:
sents_data = json.load(open(target_path + fns_jsons[20], "r"))
sents_data[100:103]

In [ ]:
import os, json, pickle
prev_path   = target_path  # where your old .pickle / .json live
target_path = "../data/sents_data_jsons_dicts/"
os.makedirs(target_path, exist_ok=True)

def token_tuple_to_dict(tok):
    """
    Accepts token tuples of length 4 or 5:
      4: (text, lemma, pos, (start, end))
      5: (text, lemma, pos, (start, end), ref_dict)
    Returns a JSON-serializable dict.
    """
    if len(tok) < 4:
        raise ValueError(f"Unexpected token shape: {tok}")

    token_text, lemma, pos, span = tok[0], tok[1], tok[2], tok[3]
    if not isinstance(span, (list, tuple)) or len(span) != 2:
        raise ValueError(f"Bad span in token: {tok}")

    char_start, char_end = int(span[0]), int(span[1])

    # Optional ref at index 4
    ref = tok[4] if len(tok) >= 5 else None

    # Make sure ref is JSON-friendly
    if isinstance(ref, dict):
        page = ref.get("page")
        # convert sets/tuples to lists to be JSON-serializable
        if isinstance(page, (set, tuple)):
            page = list(page)
        ref = {
            "page": page,
            "textblock": ref.get("textblock") or ref.get("texblock")  # tolerate earlier key name
        }
    elif ref is not None:
        # unexpected type → stringify to avoid JSON errors
        ref = str(ref)

    return {
        "token_text": token_text,
        "lemma": lemma,
        "pos": pos,
        "ref": ref,                  # << stays None if we don’t have it
        "char_start": char_start,
        "char_end": char_end,
    }

def sent_tuple_to_dict(entry):
    """
    entry is typically: (work_id, sent_id, sent_text, tokens_list)
    """
    if len(entry) != 4:
        raise ValueError(f"Unexpected sentence shape: {type(entry)} {entry}")

    work_id, sent_id, sent_text, tokens_list = entry
    tokens_dicts = [token_tuple_to_dict(tok) for tok in tokens_list]
    return {
        "work_id": work_id,
        "sent_id": int(sent_id),
        "sent_text": sent_text,
        "tokens_data": tokens_dicts,
    }

def load_any(path):
    """
    Load .pickle OR .json produced by your previous run.
    Must return a list of sentence entries (tuples/lists).
    """
    if path.endswith(".pickle"):
        with open(path, "rb") as f:
            return pickle.load(f)
    elif path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        raise ValueError(f"Unsupported file type: {path}")

for fn in os.listdir(prev_path):
    if not (fn.endswith(".pickle") or fn.endswith(".json")):
        continue

    # Keep your 6-char doc id convention if you like
    doc_id = fn[:6]
    out_path = os.path.join(target_path, f"{doc_id}.json")
    if os.path.exists(out_path):
        continue

    try:
        prev_data = load_any(os.path.join(prev_path, fn))
        # If the previous version stored only (sent_text, tokens) per sentence,
        # reconstruct work_id/sent_id here as needed:
        # e.g., prev_data == [(sent_text, tokens), ...]
        if prev_data and len(prev_data[0]) == 2:
            # synthesize (work_id, sent_id, sent_text, tokens)
            prev_data = [(doc_id, i, s[0], s[1]) for i, s in enumerate(prev_data)]

        sents_dicts = [sent_tuple_to_dict(row) for row in prev_data]

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(sents_dicts, f, ensure_ascii=False, indent=2)

        print("wrote:", out_path)

    except Exception as e:
        print("failed:", fn, "-", e)

In [ ]:
lemmatized_sents_path = "/srv/data/tome/tome-corpus/lemmatized_sents_v4-0/"
try:
    os.mkdir(lemmatized_sents_path)
except:
    pass

In [ ]:
os.listdir(lemmatized_sents_path)

In [ ]:
for fn in fns_jsons:
    lemmatized_sents = []
    sents_data = json.load(open(target_path + fn, "rb"))
    print(fn)
    for (doc_id, sent_id, sent_text, sent_data) in sents_data:
        lemmasent = []
        for wordform, lemma, tag, position, t_ref in sent_data:
            if tag in ["NOUN", "PROPN", "ADJ", "VERB"]:
                lemmasent.append(lemma.lower())
        lemmatized_sents.append(" ".join(lemmasent) + "\n")
    with open(lemmatized_sents_path + fn.replace(".json", ".txt"), "w", encoding="utf-8") as f:
        f.writelines(lemmatized_sents)

In [ ]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/lemmatized_sents_v4-0/", "../data/lemmatized_sents", dirs_exist_ok=True)
shutil.copytree("/srv/data/tome/tome-corpus/sents_data_id_jsons_v4-0/", "../data/sents_data", dirs_exist_ok=True)